# One-time Setup (`setup_nbk`)

Builds everything the query pipeline depends on, using the `optimized_*` family only.
Run top-to-bottom **once per environment** (or whenever the full source data changes).
For adding new chunks later, use `ingest_nbk` instead.

All paths/table names come from `scripts/settings.py`.

Steps: 0) download models  1) preprocess -> optimized_rag_chunks  2) Vector Search index  3) BM25 tables


In [ ]:
# --- Make scripts/ importable ---
# In Databricks Repos the repo root is on sys.path; locate the scripts/ subdir there
# (works the same when running from the repo root locally).
import os, sys

_cands = [os.path.join(p, "scripts") for p in (list(sys.path) + [os.getcwd()])]
for _cand in _cands:
    if os.path.isdir(_cand) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import settings
from lake_io import read_chunks_jsonl, read_embeddings_npy

## 0. Download models to UC Volume
Embedding model + cross-encoder reranker. Target dirs come from settings.

In [ ]:
from huggingface_hub import snapshot_download
import os

MODELS = [
    (settings.EMBED_MODEL_HF, settings.EMBED_MODEL_DIR),
    (settings.RERANKER_MODEL_HF, settings.RERANKER_MODEL_DIR),
]
for repo_id, local_dir in MODELS:
    os.makedirs(local_dir, exist_ok=True)
    snapshot_download(repo_id=repo_id, local_dir=local_dir, local_dir_use_symlinks=False)
    print("Downloaded:", repo_id, "->", local_dir)
    display(dbutils.fs.ls(local_dir))

## 1. Preprocess: build `optimized_rag_chunks`
Reads chunk text + precomputed embeddings from the lake (settings globs) and writes the three optimized_* tables.

In [ ]:
display(dbutils.fs.ls(settings.LAKE_BASE + "optimized_chunks/"))
display(dbutils.fs.ls(settings.LAKE_BASE + f"optimized_embeddings/{settings.EMBED_MODEL_NAME}/"))

In [ ]:
df_chunks = read_chunks_jsonl(spark)
df_emb = read_embeddings_npy(spark)

df_emb.write.mode("overwrite").saveAsTable(settings.EMBEDDINGS_TABLE)
df_chunks.write.mode("overwrite").saveAsTable(settings.CHUNKS_TEXT_TABLE)

df_final = df_chunks.join(df_emb, on="chunk_id", how="inner")
df_final.write.mode("overwrite").saveAsTable(settings.RAG_CHUNKS_TABLE)

In [ ]:
from pyspark.sql.functions import count, countDistinct

print("chunks_text:", spark.table(settings.CHUNKS_TEXT_TABLE).count())
print("embeddings :", spark.table(settings.EMBEDDINGS_TABLE).count())
print("rag_chunks :", spark.table(settings.RAG_CHUNKS_TABLE).count())

spark.table(settings.EMBEDDINGS_TABLE) \
    .select(count("*").alias("rows"), countDistinct("chunk_id").alias("unique_chunk_id")).show()

## 2. Vector Search index `optimized_rag_chunks_vs`

Create a Databricks Vector Search Delta Sync index on `settings.RAG_CHUNKS_TABLE` using the
precomputed `embedding` column. Set `settings.VS_ENDPOINT` (or `TDIS_VS_ENDPOINT`) first.

In [ ]:
# from databricks.vector_search.client import VectorSearchClient
# vsc = VectorSearchClient()
# vsc.create_delta_sync_index(
#     endpoint_name=settings.VS_ENDPOINT,
#     index_name=settings.VS_INDEX_FQN,
#     source_table_name=settings.RAG_CHUNKS_TABLE,
#     primary_key=settings.VS_PRIMARY_KEY,
#     embedding_dimension=settings.EMBED_DIM,
#     embedding_vector_column=settings.VS_EMBED_COL,
#     pipeline_type="TRIGGERED",
# )

## 3. Build BM25 inverted-index tables
Tokenizes `optimized_rag_chunks.text` and builds the four optimized_kw_* tables.

In [ ]:
from ingest import rebuild_bm25

print(rebuild_bm25(spark))
print("meta:", spark.table(settings.KW_META_TABLE).first())